In [1]:
try:
    from einops import rearrange
except:
    !pip install /kaggle/input/einops-0-7-0/einops-0.7.0-py3-none-any.whl

Processing /kaggle/input/einops-0-7-0/einops-0.7.0-py3-none-any.whl


In [2]:
import sys
pandarallel_path = '/kaggle/input/pandarallel-1-6-5/pandarallel-1.6.5'
if pandarallel_path not in sys.path:
    sys.path.insert(0, pandarallel_path)
    
torch_geometric_path = '/kaggle/input/torch-geometric/torch_geometric-2.3.1'
if torch_geometric_path not in sys.path:
    sys.path.insert(0, torch_geometric_path)

In [3]:
# Change image size limit before import opencv
import os
os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = pow(2,40).__str__()

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from PIL import Image
Image.MAX_IMAGE_PIXELS = pow(2,40)

import gc
import cv2
import timm
import glob
from matplotlib import pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.measure import label, regionprops, regionprops_table
# from skimage import morphology
from pandarallel import pandarallel
import random
pandarallel.initialize(use_memory_fs=False, nb_workers=os.cpu_count(), progress_bar=True)
pd.set_option('display.max_colwidth', None)

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [4]:
def fix_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    pl.seed_everything(seed, workers=True)

In [5]:
fix_seed(5)

In [6]:
threshold = 0.45  

tma_path_check = True  

debug = False 

track = 'train' if debug else 'test'

In [7]:
BASE_PATH = f'/kaggle/input/UBC-OCEAN/'
# BASE_PATH = ''

In [8]:
# checkpoint_paths = glob.glob('/kaggle/input/ubc-model-softsum/efficientnet_b4_1103-2306/*.ckpt')  # 0.48
checkpoint_paths = glob.glob('/kaggle/input/eva02-large-patch14-448-mim-m38m-ft-in22k-in1k-v7/*.ckpt')  # 
checkpoint_paths

['/kaggle/input/eva02-large-patch14-448-mim-m38m-ft-in22k-in1k-v7/fold3-epoch8-val_loss0.2292-val_acc0.8477.ckpt',
 '/kaggle/input/eva02-large-patch14-448-mim-m38m-ft-in22k-in1k-v7/fold4-epoch8-val_loss0.2381-val_acc0.7479.ckpt',
 '/kaggle/input/eva02-large-patch14-448-mim-m38m-ft-in22k-in1k-v7/fold0-epoch6-val_loss0.2296-val_acc0.8336.ckpt',
 '/kaggle/input/eva02-large-patch14-448-mim-m38m-ft-in22k-in1k-v7/fold1-epoch4-val_loss0.2327-val_acc0.8976.ckpt',
 '/kaggle/input/eva02-large-patch14-448-mim-m38m-ft-in22k-in1k-v7/fold2-epoch8-val_loss0.2261-val_acc0.8909.ckpt']

In [9]:
# get args for the image_size
args = torch.load(checkpoint_paths[0], map_location='cpu')['hyper_parameters']['args']
args

Namespace(seed=42, model='eva02_large_patch14_448.mim_m38m_ft_in22k_in1k', lr=2e-05, weight_decay=0.01, log_dir='/mnt/md0/ubc_ocean/Team_CLS/TRY5_CLS/eva02_large_patch14_448.mim_m38m_ft_in22k_in1k/v7/', num_workers=6, epochs=10, batch_size=2, val_batch_size=4, accumulate_grad_batches=1, gpus='0', patience=100, precision='16-mixed', scheduler='onecycle', n_splits=5, gem_p=3.0, dropout=0.5, repeats=2, num_tiles=8, num_tiles_val=48, tile_size=448, tile_overlap=True, add_other_cls=False, new_augment=False, data_path='/mnt/md0/ubc_ocean/ubc_small_fix/', use_std=False, loss='multilabel', dice_weight=0.3, focal_weight=0.7, entropy_weight=0.2, warmup_steps=576, run_folds=[4, 3], pseudo_threshold=0.3, num_classes=5)

In [10]:
from functools import reduce as f_reduce

def factors(n):    
    return set(f_reduce(list.__add__, 
                ([i, n//i] for i in range(1, int(n**0.5) + 1) if n % i == 0)))


def get_values(n):
    f = np.sort(list(factors(n)))
    n = len(f)
    n1 = n//2 if n%2==0 else (n+1)//2
    f = np.array([f[:(n+1)//2], f[n//2:][::-1]])
    loc = f[:, f.sum(axis=0).argmin()].tolist()
    return loc

In [11]:
def plot_sample(dst, idx=None):
    if idx is None:
        idx = np.random.randint(len(dst), size=(1,))[0]
    
    sample = dst[idx]
    images = sample['image']
    if images.shape[1] == 3:
        images = images.permute(0,2,3,1)
    
    r, c = get_values(images.shape[0]) # get factors
    
    _, axes = plt.subplots(r, c, figsize=(c*5, r*5))
    axes = axes.flatten()
    
    title = str(sample['image_id'])
    if 'label' in sample.keys():
        title += f': {cls_map_inv[sample["label"]]}'
        
    for k in range(images.shape[0]):
        image = images[k]
        if images.shape[0] == 3:
            image = image.permute(1,2,0)
        axes[k].imshow(image)
        axes[k].set_title(f"{title} : p={sample['probs'][k]: 0.4f} : s={sample['stds'][k]: 0.4f}")

In [12]:
def custom_normalize(image, **params):  
    return image / 255 

def get_transforms(debug=False):
    augmentations = [
#         A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=0),
#         A.CenterCrop(height=image_size, width=image_size),
#         A.PadIfNeeded(min_height=2*image_size, min_width=2*image_size, border_mode=0),
#         A.CenterCrop(height=2*image_size, width=2*image_size),
#         A.Resize(height=image_size, width=image_size),
    ]
    if not debug:
        augmentations.append(
            A.HorizontalFlip(p=0.5),
        )
        
    augmentations.append(
        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225), 
            max_pixel_value=255.0, 
        ),  # using basic imagenet statistics (default)   
    )

    augmentations.extend([
        ToTensorV2(),
    ])
    
    transforms = A.Compose(augmentations)
    return transforms

In [13]:
from functools import wraps
import gc

def flush_and_gc(f):
    @wraps(f)
    def g(*args, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()
        return f(*args, **kwargs)
    return g

In [14]:
import torch.nn.functional as F
from torch.nn.parameter import Parameter


def gem(x, p=1, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=1, eps=1e-6, flatten=True):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        ret = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            return ret.flatten(1)
        return ret

    def __repr__(self):
        return (
                self.__class__.__name__
                + "("
                + "p="
                + "{:.4f}".format(self.p.data.tolist()[0])
                + ", "
                + "eps="
                + str(self.eps)
                + ")"
        )

In [15]:
class Attention(nn.Module):
    def __init__(self, in_dim, hidden_dim, pool_dim=1):
        super().__init__()
        self.pool_dim = pool_dim
        self.attention = nn.Sequential(nn.Linear(in_dim, hidden_dim),
                                       nn.Tanh(),
                                       nn.Linear(hidden_dim, 1),
                                       nn.Softmax(dim=self.pool_dim))

    def forward(self, x):
        weights = self.attention(x)
        context = torch.sum((weights * x), dim=self.pool_dim)
        return context

In [16]:
def init_layer(layer):
    nn.init.xavier_uniform_(layer.weight)

    if hasattr(layer, "bias"):
        if layer.bias is not None:
            layer.bias.data.fill_(0.)
            
class TransformerBlock(nn.Module):
    def __init__(self, num_features, num_heads, dropout_rate, forward_expansion):
        super(TransformerBlock, self).__init__()
        self.attention = nn.MultiheadAttention(embed_dim=num_features, num_heads=num_heads, dropout=dropout_rate)
        self.norm1 = nn.LayerNorm(num_features)
        self.norm2 = nn.LayerNorm(num_features)

        self.feed_forward = nn.Sequential(
            nn.Linear(num_features, forward_expansion * num_features),
            nn.ReLU(),
            nn.Linear(forward_expansion * num_features, num_features)
        )
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, value, key, query):
        attention = self.attention(query, key, value)[0]
        x = self.norm1(attention + query)
        forward = self.feed_forward(x)
        out = self.norm2(forward + x)
        return out
    
class TileModel(nn.Module):
    def __init__(self, model_name='resnet34',
                 num_classes=6, dropout=0.0, gem_p=1.0, num_heads=16, forward_expansion=4):
        super().__init__()
        
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0, global_pool='')
        
        self.num_features = self.backbone.num_features

        # Transformer block
        self.transformer_block = TransformerBlock(
            num_features=self.num_features, 
            num_heads=num_heads, 
            dropout_rate=dropout, 
            forward_expansion=forward_expansion
        )

        # Dropout layer
        self.dropout = nn.Dropout(p=dropout)

        # Fully connected layer for classification
        self.fc = nn.Linear(self.num_features, num_classes)
        init_layer(self.fc)

    def forward(self, x):
        features = self.backbone(x)

        #print("Features from backbone shape", features.shape)
        # Apply Transformer block
        transformed = self.transformer_block(features, features, features)

        # Pooling and classification
        pooled_output = transformed.max(dim=1)[0]
        pooled_output = self.dropout(pooled_output)
        output = self.fc(pooled_output)

        return output
    
    def wrapDataParallel(self):
        self.backbone = nn.DataParallel(self.backbone)

In [17]:
def add_filepath(image_id):
    thumbnail_path = f"{BASE_PATH}{track}_thumbnails/{image_id}_thumbnail.png"
    real_path = f"{BASE_PATH}{track}_images/{image_id}.png"
    if os.path.exists(thumbnail_path):
        real_path = thumbnail_path
    return real_path

In [18]:
df_test = pd.read_csv(f'{BASE_PATH}{track}.csv')

if debug:
    df_test = df_test[:100]
    
df_test['img_path'] = df_test['image_id'].apply(add_filepath)

In [19]:
# change all tma from thumbnail to actual
if tma_path_check:
    df_test['img_path'] = df_test.apply(
        lambda row: f"{BASE_PATH}{track}_images/{row['image_id']}.png" if (row['image_width']==row['image_height']) else row['img_path'],
        axis=1,
    )

In [20]:
df_test

,image_id,image_width,image_height,img_path
0,41,28469,16987,/kaggle/input/UBC-OCEAN/test_thumbnails/41_thumbnail.png


In [21]:
import numpy as np
import cv2
import gc
from einops import repeat


def get_mask(image):
    # using BGR2GRAY on RGB work best for the tma
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#     gray = rgb2gray(image)
    
    # get corner values for segmentation
    h, w = gray.shape
    h1 = h//100
    w1 = w//100
    corner_values = np.mean([gray[:h1, :w1], gray[-h1:, :w1], gray[:h1, -w1:], gray[-h1:, -w1:]], axis=0).mean()
    
    # get the mask
    if corner_values > 128:
        corner_values = 255 - corner_values
        gray = 255 - gray

    mask = np.abs(gray-corner_values) > corner_values
    return mask.astype('uint8')


def get_patch_bbox(mask, CROP_SIZE=224, OVERLAP=64):
    H, W = mask.shape
    bboxes = []
    probs = []
    for y in range(0, H-CROP_SIZE+1, CROP_SIZE-OVERLAP):
        for x in range(0, W-CROP_SIZE+1, CROP_SIZE-OVERLAP):
            bbox = [x, y, x+CROP_SIZE, y+CROP_SIZE]
            prob = mask[bbox[1]:bbox[3], bbox[0]:bbox[2]].mean()
            if prob != 0:
                bboxes.append(bbox)
                probs.append(prob)
    return np.array(bboxes), np.array(probs)


def get_patch_bbox(img_path, CROP_SIZE=224, OVERLAP=64): #, include_std=False):
    img = np.array(Image.open(img_path))
    mask = get_mask(img)
    
    H, W = mask.shape
    bboxes = []
    probs = []
    stds = []
    for y in range(0, H-CROP_SIZE+1, CROP_SIZE-OVERLAP):
        for x in range(0, W-CROP_SIZE+1, CROP_SIZE-OVERLAP):
            bbox = [x, y, x+CROP_SIZE, y+CROP_SIZE]
            prob = mask[bbox[1]:bbox[3], bbox[0]:bbox[2]].mean()
            if  prob > 0.0: 
                
#                 x = np.ma.array(
#                     img[bbox[1]:bbox[3], bbox[0]:bbox[2]], 
#                     mask=repeat((mask[bbox[1]:bbox[3], bbox[0]:bbox[2]]==0), 'h w -> h w c', c=3),
#                 )
#                 std_value = x.std(axis=(0,1)).mean()

                img_ = img[bbox[1]:bbox[3], bbox[0]:bbox[2]]
                mask_ = mask[bbox[1]:bbox[3], bbox[0]:bbox[2]]
                std_value = img_[mask_.astype('bool')].std(axis=0).mean()
            
                stds.append(std_value)
                bboxes.append(bbox)
                probs.append(prob)
                
    return np.array(bboxes), np.array(probs), np.array(stds)


def get_all_patches(img_path, tile_size=args.tile_size, overlap=args.tile_overlap):
    CROP_SIZE = tile_size
    OVERLAP = CROP_SIZE//4 if overlap else 0
    
    result = dict()
    bboxes, probs, stds = get_patch_bbox(img_path, CROP_SIZE=CROP_SIZE, OVERLAP=OVERLAP)
    result.update({
        f'bboxes{CROP_SIZE}': bboxes,
        f'probs{CROP_SIZE}': probs,
        f'stds{CROP_SIZE}': stds,
    })
    return result

In [22]:
%%time

results = df_test.img_path.parallel_apply(lambda img_path: get_all_patches(img_path, tile_size=args.tile_size, overlap=args.tile_overlap))
df_test = pd.concat([df_test, pd.DataFrame.from_records(results)], axis=1)
count = df_test[df_test.columns[df_test.columns.str.startswith('bboxes')]].applymap(len)
count.columns = [column.replace('bboxes', 'count') for column in count]
df_test = pd.concat([df_test, count], axis=1)
df_test.head(1)

CPU times: user 35.3 ms, sys: 62.3 ms, total: 97.6 ms
Wall time: 1.31 s


,image_id,image_width,image_height,img_path,bboxes448,probs448,stds448,count448
0,41,28469,16987,/kaggle/input/UBC-OCEAN/test_thumbnails/41_thumbnail.png,"[[0, 0, 448, 448], [336, 0, 784, 448], [672, 0, 1120, 448], [1008, 0, 1456, 448], [1344, 0, 1792, 448], [1680, 0, 2128, 448], [2016, 0, 2464, 448], [2352, 0, 2800, 448], [0, 336, 448, 784], [336, 336, 784, 784], [672, 336, 1120, 784], [1008, 336, 1456, 784], [1344, 336, 1792, 784], [1680, 336, 2128, 784], [2016, 336, 2464, 784], [2352, 336, 2800, 784], [0, 672, 448, 1120], [336, 672, 784, 1120], [672, 672, 1120, 1120], [1008, 672, 1456, 1120], [1344, 672, 1792, 1120], [1680, 672, 2128, 1120], [2016, 672, 2464, 1120], [2352, 672, 2800, 1120], [0, 1008, 448, 1456], [336, 1008, 784, 1456], [672, 1008, 1120, 1456], [1008, 1008, 1456, 1456], [1344, 1008, 1792, 1456], [1680, 1008, 2128, 1456], [2016, 1008, 2464, 1456], [2352, 1008, 2800, 1456]]","[0.11380440848214286, 0.28122508769132654, 0.47245695153061223, 0.6705845424107143, 0.8327238121811225, 0.9856206154336735, 0.7822315449617347, 0.2088349011479592, 0.7294523278061225, 0.9806680484693877, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8480747767857143, 0.5679707429846939, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5571338887117347, 0.9160405373086735, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[44.943784038977945, 50.05601144418889, 33.48604584207231, 36.420329598178455, 41.20066227534779, 33.66829167404618, 40.167658640324156, 44.723210283447685, 33.10283607088878, 42.2512826387742, 26.598727583842926, 26.286704596112433, 24.792410585134533, 25.796730769775763, 23.57486342336073, 30.854755583591572, 35.23229950684887, 45.983162250137894, 36.40486943205368, 29.49413672585186, 30.558224468032392, 31.75240846436678, 26.1159711934844, 24.54550716498096, 39.235134979272736, 45.72036868382236, 47.8089125293302, 47.18771022745443, 39.03823732619304, 35.976298682187284, 33.44862543249359, 36.45460076874294]",32


In [23]:
gc.collect()

66

In [24]:
class UBCDatasetTest(Dataset):
    def __init__(self, df, transforms=None, tile_size=224, num_tiles=16, use_std=False):
        #assert tile_size in [224, 256, 384, 512]
        self.ids = df['image_id'].tolist()
        self.img_paths = df['img_path'].tolist()
        self.bboxes = df[f'bboxes{tile_size}']
        self.probs = df[f'probs{tile_size}']
        self.stds = df[f'stds{tile_size}']
        self.use_std = use_std
            
        self.num_tiles = num_tiles
        self.transforms = transforms
        
    def __len__(self):
        return len(self.ids)
    
    def __getitem__(self, idx):
        
        _id = self.ids[idx]
        img_path = self.img_paths[idx]
        img = cv2.imread(img_path,cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        scores = self.probs[idx]
        if self.use_std:
#             scores = self.probs[idx] * self.stds[idx]
            scores = self.probs[idx] + (self.stds[idx]/100).clip(0,1)
        
        sort_order = scores.argsort()[::-1]
        mask = (self.probs[idx] > 0)[sort_order]
        valid_sort = sort_order[mask]
        n_valid = len(valid_sort)
        n = valid_sort[:min(n_valid, self.num_tiles)]
        
        bboxes = self.bboxes[idx][n]
        
        img = np.stack([img[bbox[1]:bbox[3], bbox[0]:bbox[2]] for bbox in bboxes])
        
        if self.transforms is not None:
            img = torch.stack([self.transforms(image=imgk)['image'] for imgk in img])
        
        sample = dict(
            image_id=_id,
            image=img,
            probs=self.probs[idx][n],
            stds=self.stds[idx][n],
        )
            
        return sample

In [25]:
dst = UBCDatasetTest(df_test, tile_size=args.tile_size, num_tiles=args.num_tiles)
len(dst)

1

In [26]:
"""if track == 'test':
    plot_sample(dst, 0)
else:
    plot_sample(dst)"""

"if track == 'test':\n    plot_sample(dst, 0)\nelse:\n    plot_sample(dst)"

In [27]:
"""del dst
gc.collect()"""

'del dst\ngc.collect()'

In [28]:
# define a custom collate function for efficient merging of samples
def collate_fn(original_batch):
#     ['image_id', 'image', 'probs', 'stds', 'label']
    image_id = torch.tensor([sample['image_id'] for sample in original_batch])
    image = torch.cat([sample['image'] for sample in original_batch], dim=0)
    # create a tensor for saving the samples identifier
    index = torch.cat([torch.ones(sample['image'].shape[0])*k for k, sample in enumerate(original_batch)], dim=-1)
    
    batch = dict(
        image_id=image_id,
        image=image,
        index=index.long(),
    )
    if 'label' in original_batch[0].keys():
        label = torch.tensor([sample['label'] for sample in original_batch]).float()
        batch['label'] = label
    return batch

In [29]:
#soft summing module for tiles aggregation
from torch_geometric.utils.scatter import scatter  # to encourage unequal number of tiles


def soft_sum(x, dim, index, a=1.0):
    # find the parametric softmax indexing/probablity
    y = torch.exp(a*x)
    ym = scatter(y, index=index, dim=dim, reduce='sum')[index]
    prob = y / ym
    # multiply by prob and then sum
    x1 = scatter(x * prob, index=index, dim=0, reduce='sum')
    return x1


class SoftSum(nn.Module):
    def __init__(self, a=1.0):
        super().__init__()
        self.a = Parameter(torch.ones(1) * a)
        
    def forward(self, x, index, dim=0):
        # find the parametric softmax indexing/probablity
        y = torch.exp(self.a*x)
        ym = scatter(y, index=index, dim=dim, reduce='sum')[index]
        prob = y / ym
        # multiply by prob and then sum
        x1 = scatter(x * prob, index=index, dim=0, reduce='sum')
        return x1

class SoftSumMod(nn.Module):
    def __init__(self, a=1.0):
        super().__init__()
        self.a = a #Parameter(torch.ones(1) * a)
        
    def forward(self, x, index, dim=0):
        # find the parametric softmax indexing/probablity
        y = torch.exp(self.a*x)
        ym = scatter(y, index=index, dim=dim, reduce='sum')[index]
        prob = y / ym
        # multiply by prob and then sum
        x1 = scatter(x * prob, index=index, dim=0, reduce='sum')
        return x1
    
class CustomAggregator(nn.Module):
    def __init__(self):
        super(CustomAggregator, self).__init__()
        
        self.soft_sum_mod = SoftSumMod()
    
    def forward(self, x, index, is_training, dim=0):
        x1 = self.soft_sum_mod(x, index, dim)
        
        return x1

In [30]:
from einops import rearrange, reduce
    

class UBCModelTest(pl.LightningModule):
    def __init__(self, args):
        super().__init__()
        self.save_hyperparameters()
        
        self.net = TileModel(args.model, args.num_classes, args.dropout, args.gem_p)

        self.aggregate = CustomAggregator()
        self.activation = {
            'multilabel': nn.Sigmoid(),
            'multiclass': nn.Softmax(dim=-1),
        }[args.loss]
    
    def forward(self, image, index):
        x = self.net(image)
        x = self.aggregate(x, index, self.training)
        out = self.activation(x)
        return out

    def wrapDataParallel(self):
        self.net.wrapDataParallel()


class PredictionModel(pl.LightningModule):
    def __init__(self, ckpt_paths):
        super().__init__()
        if isinstance(ckpt_paths, str):
            ckpt_paths = [ckpt_paths]
        
        self.models = nn.ModuleList([
            UBCModelTest.load_from_checkpoint(ckpt_path, map_location='cpu', strict=False) for ckpt_path in ckpt_paths
        ])
        
        for model in self.models:
            model.wrapDataParallel()
        
    @flush_and_gc
    def forward(self, image, index): 
        out = torch.stack([model(image, index) for model in self.models], dim=0).mean(dim=0)
        return out
    
    def predict_step(self, batch, batch_idx=0):
        image_id = batch['image_id'].cpu().numpy()
        image = batch['image']
        index = batch['index']
        
        pred = self(image, index)
        
        return {'image_id': image_id, 'pred': pred.cpu().numpy()}

In [31]:
# cls_map_inv = {index: key for index, key in enumerate(['CC', 'EC', 'HGSC', 'LGSC', 'MC'])}
cls_map_inv = {0: 'HGSC', 1: 'EC', 2: 'CC', 3: 'LGSC', 4: 'MC'}


def decode_result(pred, threshold=threshold):
    index = pred.argmax()
    value = pred.max()
    
    if args.loss == 'multilabel':
        if value < threshold:
            return 'Other'
    else:
        pnorm = np.linalg.norm(pred)
        if pnorm < threshold:
            return 'Other'
    
    return cls_map_inv[index]

In [32]:

dst_test = UBCDatasetTest(
    df_test, 
    transforms=get_transforms(),
    tile_size=args.tile_size,
    num_tiles=args.num_tiles_val,
    use_std=args.use_std,
)

test_loader = DataLoader(
    dst_test, 
#     batch_size=1,
    batch_size=2,
    shuffle=False,
    num_workers=os.cpu_count(),
    collate_fn=collate_fn,
)

In [33]:
model = PredictionModel(checkpoint_paths)

In [34]:
trainer = pl.Trainer(
    accelerator='gpu',
    devices=1, #[int(t) for t in args.gpus.split(',')],
    logger=None,
    precision=args.precision,
)

# Train the model
preds = trainer.predict(model, dataloaders=test_loader)

Predicting: 0it [00:00, ?it/s]

In [35]:
del model, trainer
torch.cuda.empty_cache()
gc.collect()

47230

In [36]:
submission = pd.DataFrame(preds).explode(column=['image_id', 'pred'], ignore_index=True)
submission['label'] = submission['pred'].apply(decode_result)
submission.head()

,image_id,pred,label
0,41,"[0.7093086, 0.20415829, 0.3717823, 0.34952018, 0.29725334]",HGSC


In [37]:
if track == 'train':
    submission.to_csv("train_pred.csv", index=False)

In [38]:
submission.drop(columns=['pred'], inplace=True)
submission.head()

,image_id,label
0,41,HGSC


In [39]:
submission.to_csv("submission.csv", index=False)